# Week 12: Decoupled Knowledge Distillation (DKD)

**Paper:** Zhao *et al.*, *Decoupled Knowledge Distillation*, CVPR 2022 ([arXiv:2203.08679](https://arxiv.org/abs/2203.08679)).

Vanilla KD matches the full teacher softmax with one KL term. **DKD splits** the signal into:

- **TCKD (target-class KD):** a 2-way distribution — probability mass on the **true class** vs **all non-target classes** — and matches teacher vs student on that **binary** split ("sample difficulty").
- **NCKD (non-target KD):** masks the ground-truth logit (large negative), then **KL on the remaining (wrong) classes** — the classic **dark knowledge** among distractors, decoupled from the target logit.

Total distillation term: **`α · TCKD + β · NCKD`** (separate from CE). Implementation follows the reference **[mdistiller `DKD.py`](https://github.com/megvii-research/mdistiller/blob/master/mdistiller/distillers/DKD.py)**.

**Setup:** same as Week 11 — frozen **pretrained ResNet-56** CIFAR-10 teacher (`torch.hub`), ViT student, metric = **wall-clock to 80%** test accuracy.

## Imports

In [ ]:
import sys
import math
import time
from pathlib import Path
from typing import Dict, List, Optional

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / "data"

import torch
import torch.nn as nn
import torch.nn.functional as F

from src.model import count_parameters
from src.utils import (
    get_device,
    get_cifar10_loaders,
    get_model,
    set_seed,
    validate,
    get_peak_gpu_memory_mb,
    reset_peak_gpu_memory,
)

torch.set_float32_matmul_precision("high")
print(torch.__version__, get_device())

## Config

DKD hyperparameters (`ALPHA_DKD`, `BETA_DKD`, `TEMP_DKD`) are the paper’s decoupled weights and temperature. **`CE_WEIGHT`** scales the hard-label cross-entropy. Optional **`DKD_WARMUP_EPOCHS`** ramps the DKD term from 0 → full over the first epochs (as in mdistiller).

In [ ]:
SEED = 42
TARGET_ACC = 80.0
MAX_EPOCHS = 100

BATCH_SIZE = 512
LR_REF_BATCH = 512
LR_REF = 1e-3
LR = LR_REF * (BATCH_SIZE / LR_REF_BATCH)
NUM_WORKERS = 4
WARMUP_EPOCHS = 5

# DKD (typical CIFAR-scale settings; tune α, β, T)
TEMP_DKD = 4.0
ALPHA_DKD = 1.0   # weight on TCKD
BETA_DKD = 8.0    # weight on NCKD
CE_WEIGHT = 1.0
DKD_WARMUP_EPOCHS = 20  # ramp dkd; set 0 or 1 to disable ramp

VALIDATE_EVERY_COARSE = 2
VALIDATE_DENSE_AFTER = 74.0

set_seed(SEED, deterministic=False)
print(f"batch={BATCH_SIZE} LR={LR:.2e} DKD T={TEMP_DKD} α={ALPHA_DKD} β={BETA_DKD} CE_w={CE_WEIGHT}")

## Data

In [ ]:
train_loader, test_loader = get_cifar10_loaders(
    data_dir=str(DATA_DIR),
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    augment_train=True,
    use_randaugment=True,
    randaugment_num_ops=2,
    randaugment_magnitude=9,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=2,
)
xb, _ = next(iter(train_loader))
device = get_device()
print(len(train_loader), "batches", xb.shape)

## DKD loss (Megvii reference)

In [ ]:
def _get_gt_mask(logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    target = target.reshape(-1)
    return torch.zeros_like(logits).scatter_(1, target.unsqueeze(1), 1).bool()


def _get_other_mask(logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    target = target.reshape(-1)
    return torch.ones_like(logits).scatter_(1, target.unsqueeze(1), 0).bool()


def cat_mask(t: torch.Tensor, mask1: torch.Tensor, mask2: torch.Tensor) -> torch.Tensor:
    """Collapse class dim to 2 bins: mass on GT vs mass on all other classes."""
    t1 = (t * mask1).sum(dim=1, keepdim=True)
    t2 = (t * mask2).sum(dim=1, keepdim=True)
    return torch.cat([t1, t2], dim=1)


def dkd_loss(
    logits_student: torch.Tensor,
    logits_teacher: torch.Tensor,
    target: torch.Tensor,
    alpha: float,
    beta: float,
    temperature: float,
) -> torch.Tensor:
    """
    Decoupled KD: α · TCKD + β · NCKD, scaled by T² / B (Megvii mdistiller).
    """
    gt_mask = _get_gt_mask(logits_student, target)
    other_mask = _get_other_mask(logits_student, target)
    pred_student = F.softmax(logits_student / temperature, dim=1)
    pred_teacher = F.softmax(logits_teacher / temperature, dim=1)
    pred_student = cat_mask(pred_student, gt_mask, other_mask)
    pred_teacher = cat_mask(pred_teacher, gt_mask, other_mask)
    log_pred_student = torch.log(pred_student.clamp_min(1e-12))
    tckd = F.kl_div(log_pred_student, pred_teacher, reduction="sum") * (temperature ** 2) / target.shape[0]

    pred_teacher_nt = F.softmax(logits_teacher / temperature - 1000.0 * gt_mask.float(), dim=1)
    log_pred_student_nt = F.log_softmax(logits_student / temperature - 1000.0 * gt_mask.float(), dim=1)
    nckd = F.kl_div(log_pred_student_nt, pred_teacher_nt, reduction="sum") * (temperature ** 2) / target.shape[0]

    return alpha * tckd + beta * nckd


print("DKD helpers ready.")

## Teacher + student

In [ ]:
device = get_device()
cnn_teacher = torch.hub.load(
    "chenyaofo/pytorch-cifar-models",
    "cifar10_resnet56",
    pretrained=True,
)
cnn_teacher.eval()
for p in cnn_teacher.parameters():
    p.requires_grad_(False)
cnn_teacher = cnn_teacher.to(device)
_, tacc = validate(cnn_teacher, test_loader, nn.CrossEntropyLoss(), device)
print(f"Teacher test {tacc:.2f}%")


def make_student():
    return get_model(patch_size=4, embed_dim=192, depth=6, num_heads=6, dropout=0.0)

print("student params", count_parameters(make_student()))

## Train: `CE_WEIGHT · CE + ramp · DKD`

In [ ]:
def train_dkd(student, teacher, train_loader, test_loader):
    device = get_device()
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True
    student = student.to(device)
    teacher = teacher.to(device)
    if device.type == "cuda":
        student = torch.compile(student, mode="reduce-overhead")

    opt = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=0.01)

    def lr_lambda(ep):
        if ep < WARMUP_EPOCHS:
            return (ep + 1) / WARMUP_EPOCHS
        p = (ep - WARMUP_EPOCHS) / max(1, MAX_EPOCHS - WARMUP_EPOCHS)
        return 0.5 * (1 + math.cos(math.pi * p))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    use_amp = device.type == "cuda"
    dtype = torch.bfloat16 if use_amp and torch.cuda.is_bf16_supported() else torch.float16

    hist: Dict[str, List] = {"test_acc": [], "train_loss": [], "wall_time": []}
    best, ttt = 0.0, None
    last_val: Optional[float] = None
    t0 = time.perf_counter()

    for ep in range(1, MAX_EPOCHS + 1):
        ramp = min(1.0, ep / max(1, DKD_WARMUP_EPOCHS))
        student.train()
        tot = 0.0
        t_ep = time.perf_counter()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            if use_amp:
                with torch.amp.autocast(device_type="cuda", dtype=dtype):
                    zs = student(x)
                with torch.no_grad():
                    with torch.amp.autocast(device_type="cuda", dtype=dtype):
                        zt = teacher(x)
            else:
                zs = student(x)
                with torch.no_grad():
                    zt = teacher(x)
            ce = F.cross_entropy(zs, y)
            dkd = dkd_loss(zs, zt.detach(), y, ALPHA_DKD, BETA_DKD, TEMP_DKD)
            loss = CE_WEIGHT * ce + ramp * dkd
            loss.backward()
            nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            opt.step()
            tot += loss.item()
        sched.step()
        wt = time.perf_counter() - t_ep

        run_val = (ep == 1) or (ep == MAX_EPOCHS)
        if last_val is not None and last_val >= VALIDATE_DENSE_AFTER:
            run_val = True
        elif ep % VALIDATE_EVERY_COARSE == 0:
            run_val = True
        if run_val:
            _, acc = validate(student, test_loader, nn.CrossEntropyLoss(), device)
            last_val = acc
        else:
            acc = last_val if last_val is not None else 0.0

        best = max(best, acc)
        if ttt is None and acc >= TARGET_ACC:
            ttt = time.perf_counter() - t0
        hist["train_loss"].append(tot / len(train_loader))
        hist["test_acc"].append(acc)
        hist["wall_time"].append(wt)
        tag = "~" if not run_val else ""
        print(f"ep{ep:3d} ramp={ramp:.2f} loss {hist['train_loss'][-1]:.4f} test{tag} {acc:.2f}% ({wt:.1f}s/ep)")
        if ttt is not None:
            print(f"  >>> {TARGET_ACC}% in {ttt:.2f}s wall-clock")
            break
    return {"history": hist, "best_acc": best, "time_to_target": ttt, "total_time": time.perf_counter() - t0}


set_seed(SEED)
reset_peak_gpu_memory()
results = train_dkd(make_student(), cnn_teacher, train_loader, test_loader)
print(results["time_to_target"], results["best_acc"], "peak", get_peak_gpu_memory_mb())

## Plot

In [ ]:
import matplotlib.pyplot as plt

h = results["history"]
cum = [sum(h["wall_time"][: i + 1]) for i in range(len(h["wall_time"]))]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(h["test_acc"], lw=2)
ax[0].axhline(TARGET_ACC, color="gray", ls="--")
ax[0].set_xlabel("epoch")
ax[0].set_ylabel("test %")
ax[0].set_title("DKD: test accuracy")
ax[0].grid(alpha=0.3)
ax[1].plot(cum, h["test_acc"], lw=2)
ax[1].axhline(TARGET_ACC, color="gray", ls="--")
if results["time_to_target"]:
    ax[1].axvline(results["time_to_target"], color="r", ls=":", label="hit")
    ax[1].legend()
ax[1].set_xlabel("wall s")
ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "notebooks" / "week12_dkd_results.png", dpi=150, bbox_inches="tight")
plt.show()